# FastTextExtractor Test Notebook

This notebook demonstrates and tests the `FastTextExtractor` strategy for document extraction.

## What we'll test:
1. Document triage and classification
2. FastTextExtractor confidence scoring
3. Content extraction (text blocks, tables, figures)
4. Cost estimation
5. Sample extracted content display

## Setup and Imports

In [1]:
import sys
from pathlib import Path
from pprint import pprint

# Add src to path
sys.path.insert(0, str(Path().absolute().parent))

from src.agents.triage import TriageAgent
from src.strategies.fast_text import FastTextExtractor
from src.models.document_profile import DocumentProfile

c:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Select Test Document

In [4]:
# Choose a test document
# Exact paths to the test documents
test_documents = {
    "class_a": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_a\CBE Annual Report 2006-7 .pdf"),
    "class_b": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_b\Annual_Report_JUNE-2023.pdf"),
    "class_c": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_c\fta_performance_survey_final_report_2022.pdf"),
    "class_d": Path(r"C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_d\tax_expenditure_ethiopia_2021_22.pdf"),
}

# Select which document to test
selected_doc = "class_a"  # Change this to test different documents: "class_a", "class_b", "class_c", or "class_d"
pdf_path = test_documents[selected_doc]

if not pdf_path.exists():
    print(f"⚠️  Document not found: {pdf_path}")
    print("Available documents:")
    for key, path in test_documents.items():
        exists = "✓" if path.exists() else "✗"
        print(f"  {exists} {key}: {path.name}")
else:
    print(f"✓ Selected document: {pdf_path.name}")
    print(f"  Path: {pdf_path}")
    print(f"  Size: {pdf_path.stat().st_size / 1024 / 1024:.2f} MB")

✓ Selected document: CBE Annual Report 2006-7 .pdf
  Path: C:\Users\Hello\OneDrive\Desktop\Tenacious-Projects\layout-aware-document-intelligence-refinery\data\class_a\CBE Annual Report 2006-7 .pdf
  Size: 34.17 MB


## Step 1: Document Triage

In [5]:
# Initialize Triage Agent
profiles_dir = Path(".refinery/profiles")
profiles_dir.mkdir(parents=True, exist_ok=True)
triage = TriageAgent(profiles_dir=profiles_dir)

# Classify the document
print("🔍 Running Triage Agent...")
profile = triage.classify_document(pdf_path)

print("\n📋 Document Profile:")
print(f"  Document ID: {profile.doc_id}")
print(f"  Origin Type: {profile.origin_type}")
print(f"  Layout Complexity: {profile.layout_complexity}")
print(f"  Domain Hint: {profile.domain_hint}")
print(f"  Estimated Cost: {profile.estimated_cost}")
print(f"  Language: {profile.language} (confidence: {profile.language_confidence:.2f})")
print(f"  Page Count: {profile.metadata.page_count}")
print(f"  File Size: {profile.metadata.size_bytes / 1024 / 1024:.2f} MB")

🔍 Running Triage Agent...

📋 Document Profile:
  Document ID: CBE Annual Report 2006-7 
  Origin Type: mixed
  Layout Complexity: table_heavy
  Domain Hint: financial
  Estimated Cost: needs_layout_model
  Language: en (confidence: 1.00)
  Page Count: 60
  File Size: 34.17 MB


## Step 2: Initialize FastTextExtractor

In [6]:
# Create extractor instance
extractor = FastTextExtractor()

print(f"✓ Initialized: {extractor.name} extractor")
print(f"\n🔍 Checking if extractor can handle this document...")
can_handle = extractor.can_handle(profile)
print(f"  Can Handle: {can_handle}")

if not can_handle:
    print("\n⚠️  Warning: FastTextExtractor may not be optimal for this document")
    print("   Consider using a layout-aware or vision-augmented strategy instead")
else:
    print("\n✓ This document is suitable for fast text extraction")

✓ Initialized: fast_text extractor

🔍 Checking if extractor can handle this document...
  Can Handle: False

⚠️  Warning: FastTextExtractor may not be optimal for this document
   Consider using a layout-aware or vision-augmented strategy instead


## Step 3: Confidence Scoring

In [7]:
print("📊 Calculating confidence score...")
confidence = extractor.confidence_score(str(pdf_path))

print(f"\n  Confidence: {confidence:.4f} ({confidence*100:.1f}%)")

# Interpret confidence
if confidence >= 0.8:
    interpretation = "Very High - Extraction should work excellently"
elif confidence >= 0.6:
    interpretation = "High - Extraction should work well"
elif confidence >= 0.4:
    interpretation = "Medium - Extraction may have some issues"
else:
    interpretation = "Low - Consider using a different strategy"

print(f"  Interpretation: {interpretation}")

📊 Calculating confidence score...

  Confidence: 0.6460 (64.6%)
  Interpretation: High - Extraction should work well


## Step 4: Cost Estimation

In [8]:
print("💰 Estimating extraction cost...")
cost = extractor.cost_estimate(str(pdf_path))

print(f"\n  Total Cost: ${cost['total_cost_usd']:.4f}")
print(f"  Cost per Page: ${cost['cost_per_page']:.4f}")
print(f"\n  💡 Fast text extraction is CPU-only and effectively free!")

💰 Estimating extraction cost...

  Total Cost: $0.0000
  Cost per Page: $0.0000

  💡 Fast text extraction is CPU-only and effectively free!


## Step 5: Extract Content

In [9]:
print("📄 Extracting content from document...")
print("   This may take a moment for large documents...")

import time
start_time = time.time()

extracted = extractor.extract(str(pdf_path))

elapsed = time.time() - start_time

print(f"\n✓ Extraction completed in {elapsed:.2f} seconds")
print(f"\n📊 Extraction Summary:")
print(f"  Text Blocks: {len(extracted.text_blocks):,}")
print(f"  Tables: {len(extracted.tables)}")
print(f"  Figures: {len(extracted.figures)}")
print(f"  Reading Order: {len(extracted.reading_order)} indices")

📄 Extracting content from document...
   This may take a moment for large documents...

✓ Extraction completed in 37.36 seconds

📊 Extraction Summary:
  Text Blocks: 17,609
  Tables: 109
  Figures: 79
  Reading Order: 17609 indices


## Step 6: Sample Text Blocks

In [10]:
print("📝 Sample Text Blocks (first 10):")
print("=" * 80)

for i, block in enumerate(extracted.text_blocks[:10], 1):
    content = block.content.strip()
    if len(content) > 60:
        content = content[:60] + "..."
    
    print(f"\n[{i}] Page {block.page_num}")
    print(f"    Content: {content!r}")
    print(f"    BBox: ({block.bbox.x0:.1f}, {block.bbox.y0:.1f}) → ({block.bbox.x1:.1f}, {block.bbox.y1:.1f})")
    print(f"    Size: {len(block.content)} chars")

📝 Sample Text Blocks (first 10):

[1] Page 1
    Content: 'M'
    BBox: (-822.5, -852.9) → (796.5, 790.9)
    Size: 1 chars

[2] Page 1
    Content: 'CBE'
    BBox: (-241.4, 804.7) → (-218.4, 816.7)
    Size: 3 chars

[3] Page 1
    Content: '-'
    BBox: (-215.6, 804.7) → (-211.6, 816.7)
    Size: 1 chars

[4] Page 1
    Content: 'Annual'
    BBox: (-208.7, 804.7) → (-172.9, 816.7)
    Size: 6 chars

[5] Page 1
    Content: 'Report'
    BBox: (-170.0, 804.7) → (-136.0, 816.7)
    Size: 6 chars

[6] Page 1
    Content: '2006-07'
    BBox: (-133.1, 804.7) → (-91.7, 816.7)
    Size: 7 chars

[7] Page 1
    Content: 'CBE'
    BBox: (89.7, 804.6) → (112.7, 816.6)
    Size: 3 chars

[8] Page 1
    Content: '-'
    BBox: (115.6, 804.6) → (119.5, 816.6)
    Size: 1 chars

[9] Page 1
    Content: 'Annual'
    BBox: (122.4, 804.6) → (158.3, 816.6)
    Size: 6 chars

[10] Page 1
    Content: 'Report'
    BBox: (161.2, 804.6) → (195.2, 816.6)
    Size: 6 chars


## Step 7: Sample Tables

In [11]:
if extracted.tables:
    print(f"📊 Sample Tables (showing first {min(3, len(extracted.tables))}):")
    print("=" * 80)
    
    for i, table in enumerate(extracted.tables[:3], 1):
        print(f"\n[Table {i}] Page {table.page_num}")
        print(f"  Headers ({len(table.headers)}): {table.headers}")
        print(f"  Rows: {len(table.rows)}")
        
        if table.rows:
            print(f"  First Row: {table.rows[0]}")
            if len(table.rows) > 1:
                print(f"  Second Row: {table.rows[1]}")
        
        print(f"  BBox: ({table.bbox.x0:.1f}, {table.bbox.y0:.1f}) → ({table.bbox.x1:.1f}, {table.bbox.y1:.1f})")
else:
    print("📊 No tables detected in this document")

📊 Sample Tables (showing first 3):

[Table 1] Page 3
  Headers (2): ['Commercial Bank\nEthiopia\nof\nAnnual Report 2006-07\nI\nCBE - Annual Report 2006-07\nCBE - Annual Report 2006-07', 'None']
  Rows: 1
  First Row: ['None', '']
  BBox: (0.0, 0.0) → (595.3, 841.9)

[Table 2] Page 5
  Headers (2): ['Commercial Bank of Ethiopia\nProfile\nThe State Bank of Ethiopia was founded in 1942 with twin objectives: performing the duties\nof both commercial and central banking. In 1963, the commercial Bank of Ethiopia (CBE)\nwas legally established as Share Company to take over the commercial banking activities of the\nstate Bank of Ethiopia. In the1974 revolution, CBE got its strength by merging with the privately\nowned Addis Ababa Bank. Since then, it has been playing a significant role in the development\nendeavor of the country.\nThe CBE, which is striving to embark into a world-class bank, is rendering state-of-the-art and\nreliable services to its millions of customers both locally and abro

## Step 8: Sample Figures

In [12]:
if extracted.figures:
    print(f"🖼️  Sample Figures (showing first {min(5, len(extracted.figures))}):")
    print("=" * 80)
    
    for i, figure in enumerate(extracted.figures[:5], 1):
        print(f"\n[Figure {i}] Page {figure.page_num}")
        print(f"  Caption: {figure.caption or '(none)'}")
        print(f"  BBox: ({figure.bbox.x0:.1f}, {figure.bbox.y0:.1f}) → ({figure.bbox.x1:.1f}, {figure.bbox.y1:.1f})")
        width = figure.bbox.x1 - figure.bbox.x0
        height = figure.bbox.y1 - figure.bbox.y0
        print(f"  Size: {width:.1f} × {height:.1f} points")
else:
    print("🖼️  No figures/images detected in this document")

🖼️  Sample Figures (showing first 5):

[Figure 1] Page 1
  Caption: (none)
  BBox: (0.0, 0.0) → (595.3, 842.0)
  Size: 595.3 × 842.0 points

[Figure 2] Page 1
  Caption: (none)
  BBox: (0.0, 0.0) → (595.3, 842.0)
  Size: 595.3 × 842.0 points

[Figure 3] Page 2
  Caption: (none)
  BBox: (0.0, 0.0) → (595.2, 841.9)
  Size: 595.2 × 841.9 points

[Figure 4] Page 3
  Caption: (none)
  BBox: (441.7, 776.7) → (595.3, 842.0)
  Size: 153.6 × 65.3 points

[Figure 5] Page 3
  Caption: (none)
  BBox: (335.2, 56.7) → (538.6, 345.8)
  Size: 203.4 × 289.1 points


## Step 9: Text Block Statistics

In [13]:
import statistics
from collections import Counter

if extracted.text_blocks:
    # Calculate statistics
    char_counts = [len(block.content) for block in extracted.text_blocks]
    pages = [block.page_num for block in extracted.text_blocks]
    
    print("📈 Text Block Statistics:")
    print("=" * 80)
    print(f"  Total Text Blocks: {len(extracted.text_blocks):,}")
    print(f"  Total Characters: {sum(char_counts):,}")
    print(f"  Average Chars per Block: {statistics.mean(char_counts):.1f}")
    print(f"  Median Chars per Block: {statistics.median(char_counts):.1f}")
    print(f"  Min Chars: {min(char_counts)}")
    print(f"  Max Chars: {max(char_counts)}")
    print(f"  Pages Covered: {min(pages)} to {max(pages)}")
    
    # Blocks per page
    blocks_per_page = Counter(pages)
    print(f"\n  Blocks per Page (sample):")
    for page_num in sorted(blocks_per_page.keys())[:10]:
        print(f"    Page {page_num}: {blocks_per_page[page_num]} blocks")

📈 Text Block Statistics:
  Total Text Blocks: 17,609
  Total Characters: 84,789
  Average Chars per Block: 4.8
  Median Chars per Block: 4.0
  Min Chars: 1
  Max Chars: 38
  Pages Covered: 1 to 60

  Blocks per Page (sample):
    Page 1: 13 blocks
    Page 3: 24 blocks
    Page 5: 343 blocks
    Page 6: 170 blocks
    Page 7: 7 blocks
    Page 9: 496 blocks
    Page 10: 417 blocks
    Page 11: 797 blocks
    Page 12: 243 blocks
    Page 13: 363 blocks


## Step 10: Full Document Text Preview

In [14]:
# Combine all text blocks into a single document text
full_text = " ".join(block.content for block in extracted.text_blocks)

print(f"📄 Full Document Text Preview:")
print("=" * 80)
print(f"\nTotal Length: {len(full_text):,} characters")
print(f"\nFirst 500 characters:")
print("-" * 80)
print(full_text[:500])
print("-" * 80)
print(f"\n... (truncated) ...")

📄 Full Document Text Preview:

Total Length: 102,397 characters

First 500 characters:
--------------------------------------------------------------------------------
M CBE - Annual Report 2006-07 CBE - Annual Report 2006-07 PB 1 CBE - Annual Report 2006-07 CBE - Annual Report 2006-07 CBE - Annual Report 2006-07 2 I Commercial Bank of Ethiopia Annual Report 2006-07 CBE - Annual Report 2006-07 CBE - Annual Report 2006-07 II 1 CBE - Annual Report 2006-07 CBE - Annual Report 2006-07 CBE - Annual Report 2006-07 II 1 Profile The State Bank of Ethiopia was founded in 1942 with twin objectives: performing the duties of both commercial and central banking. In 1963, t
--------------------------------------------------------------------------------

... (truncated) ...


## Summary

In [15]:
print("✅ Test Summary:")
print("=" * 80)
print(f"\nDocument: {pdf_path.name}")

print("\nProfile:")
print(f"  - Document ID: {profile.doc_id}")
print(f"  - Origin: {profile.origin_type}  (native_digital = has a real text layer; scanned_image = image-only; mixed = both)")
print(f"  - Layout: {profile.layout_complexity}  (multi_column = multiple text columns detected via x-position clustering)")
print(f"  - Domain: {profile.domain_hint}  (financial = keyword hits on revenue/profit/balance sheet/etc.)")
print(f"  - Estimated Cost: {profile.estimated_cost}  (needs_layout_model = multi-column / table-heavy / figure-heavy layouts)")
print(
    f"  - Language: {profile.language} (confidence: {profile.language_confidence:.2f})  "
    "(heuristic over Unicode ranges; detects Ethiopic/Amharic vs default English)"
)
print(f"  - Page Count: {profile.metadata.page_count}  (pages = len(pdf.pages))")
print(
    f"  - File Size: {profile.metadata.size_bytes / 1024 / 1024:.2f} MB  "
    "(bytes from filesystem; useful for cost/performance expectations)"
)

print("\nExtraction:")
print(f"  - Strategy: {extractor.name}  (uses pdfplumber as the primary parser for text/tables/images)")
print(f"  - Can Handle: {can_handle}  (True only for native_digital + single_column profiles)")
print(f"  - Confidence: {confidence:.2%}  (combined character-density/layout/table scores)")
print(f"  - Cost: ${cost['total_cost_usd']:.4f}  (fast text is CPU-only and effectively free)")

print("\nResults:")
print(f"  - Text Blocks: {len(extracted.text_blocks):,}")
print(f"  - Tables: {len(extracted.tables)}")
print(f"  - Figures: {len(extracted.figures)}")

print("\nDetails:")
print(
    "  - Origin type heuristics: avg_chars/page, image-to-page area ratio, "
    "and font coverage (chars layer). "
    "scanned_image if chars < 50/page AND images > 80% of page area; "
    "mixed if images are high or font coverage is low; otherwise native_digital."
)
print(
    "  - Layout complexity: estimate columns by clustering x-positions into "
    "4 bins; count table-like rows and image pages to choose between "
    "single_column, multi_column, table_heavy, and figure_heavy."
)
print(
    "  - Domain hint: keyword hits over the first ~10 pages (financial/legal/" 
    "technical/medical); best-scoring domain wins with a simple confidence "
    "scaling."
)
print(
    "  - Language: sample text from first pages; measure proportion of Ethiopic "
    "(Amharic) Unicode codepoints vs others to choose 'am' vs 'en'."
)
print(
    "  - Estimated cost: scanned_image → needs_vision_model; complex layouts "
    "(multi_column/table_heavy/figure_heavy/mixed) → needs_layout_model; "
    "simple single_column native_digital → fast_text_sufficient."
)

print("\n✓ FastTextExtractor test completed successfully!")

✅ Test Summary:

Document: CBE Annual Report 2006-7 .pdf

Profile:
  - Document ID: CBE Annual Report 2006-7 
  - Origin: mixed  (native_digital = has a real text layer; scanned_image = image-only; mixed = both)
  - Layout: table_heavy  (multi_column = multiple text columns detected via x-position clustering)
  - Domain: financial  (financial = keyword hits on revenue/profit/balance sheet/etc.)
  - Estimated Cost: needs_layout_model  (needs_layout_model = multi-column / table-heavy / figure-heavy layouts)
  - Language: en (confidence: 1.00)  (heuristic over Unicode ranges; detects Ethiopic/Amharic vs default English)
  - Page Count: 60  (pages = len(pdf.pages))
  - File Size: 34.17 MB  (bytes from filesystem; useful for cost/performance expectations)

Extraction:
  - Strategy: fast_text  (uses pdfplumber as the primary parser for text/tables/images)
  - Can Handle: False  (True only for native_digital + single_column profiles)
  - Confidence: 64.60%  (combined character-density/layout/